# Safety Helmet and Reflective Jacket Detection

- **Dataset**: Subset from [Safety Helmet and Reflective Jacket dataset](https://datasetninja.com/safety-helmet-and-reflective-jacket#license).
- **Dataset License**: Apache 2.0.
- **Model**: [NanoDet](https://github.com/RangiLyu/nanodet).
- **Task**: Object Detection.
- **Metrics**: mAP@0.5.

## 1. Data Processing

- Download the dataset from [https://datasetninja.com/safety-helmet-and-reflective-jacket](https://datasetninja.com/safety-helmet-and-reflective-jacket), extract the validation folder into `./data/helmet_jacket/raw/` folder.
- Convert to AnyLabeling format:



In [9]:
import json
from pathlib import Path

raw_data_dir = Path("./data/helmet_jacket/raw")
raw_images_dir = raw_data_dir / "img"
raw_labels_dir = raw_data_dir / "ann"

target_data_dir = Path("./data/helmet_jacket/anylb")

for image_file in raw_images_dir.glob("*.jpg"):
    label_file = raw_labels_dir / f"{image_file.stem}{image_file.suffix}.json"
    if not label_file.exists():
        print(f"Label file {label_file} does not exist")
        continue

    with open(label_file, "r") as f:
        label_data = json.load(f)

    # Convert to labelme format
    labelme_data = {
        "version": "5.1.1",
        "flags": {},
        "shapes": [],
        "imagePath": image_file.name,
        "imageData": None,
        "imageHeight": label_data["size"]["height"],
        "imageWidth": label_data["size"]["width"],
    }

    for obj in label_data["objects"]:
        shape = {
            "label": obj["classTitle"],
            "points": obj["points"]["exterior"],
            "group_id": None,
            "shape_type": "rectangle",
            "flags": {},
        }
        labelme_data["shapes"].append(shape)

    # Create target directory if it doesn't exist
    target_data_dir.mkdir(parents=True, exist_ok=True)

    # Save labelme JSON file
    target_json_file = target_data_dir / f"{image_file.stem}.json"
    with open(target_json_file, "w") as f:
        json.dump(labelme_data, f, indent=2)

    # Copy image file to target directory
    import shutil

    shutil.copy2(image_file, target_data_dir / image_file.name)

print(f"Converted {len(list(raw_images_dir.glob('*.jpg')))} images to labelme format")

Converted 1575 images to labelme format


In [14]:
import random

split_data_dir = Path("./data/helmet_jacket/anylb_split")
split_data_dir.mkdir(parents=True, exist_ok=True)

# Get all image files from the anylb directory
target_data_dir = Path("./data/helmet_jacket/anylb")
all_images = list(target_data_dir.glob("*.jpg"))
print(f"Total images: {len(all_images)}")

# Shuffle the list of images
random.seed(42)
random.shuffle(all_images)

# Calculate split sizes
total_images = len(all_images)
train_size = int(0.6 * total_images)
val_size = int(0.2 * total_images)
test_size = total_images - train_size - val_size

# Split the data
train_images = all_images[:train_size]
val_images = all_images[train_size : train_size + val_size]
test_images = all_images[train_size + val_size :]


# Function to copy images and their corresponding JSON files
def copy_files(image_list, destination):
    for img in image_list:
        shutil.copy2(img, destination)
        json_file = target_data_dir / f"{img.stem}.json"
        if json_file.exists():
            shutil.copy2(json_file, destination)


# Create split directories and copy files
for split, images in [
    ("train", train_images),
    ("val", val_images),
    ("test", test_images),
]:
    split_dir = split_data_dir / split
    split_dir.mkdir(parents=True, exist_ok=True)
    copy_files(images, split_dir)

print(f"Split {total_images} images into:")
print(f"Train: {len(train_images)} images")
print(f"Validation: {len(val_images)} images")
print(f"Test: {len(test_images)} images")

Total images: 1575
Split 1575 images into:
Train: 945 images
Validation: 315 images
Test: 315 images
